# 02 — Poisson e municípios grandes x pequenos (RQ3) · Pollyana
Suba o `painel.csv` no Colab (ícone de pasta → upload) e rode as células em ordem.

In [ ]:
import pandas as pd, numpy as np, warnings
import matplotlib.pyplot as plt
from sklearn.linear_model import PoissonRegressor, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error as mae, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import os
# funciona no repositório (notebooks/ -> ../dados/) e no Colab (painel.csv enviado)
caminho = next(c for c in ["../dados/painel.csv", "dados/painel.csv", "painel.csv"] if os.path.exists(c))
p = pd.read_csv(caminho)
p["mes"] = pd.to_datetime(p["mes"])
print(p.shape)  # esperado: (16830, 14)

feats = ["lag1","lag2","lag3","lag6","lag12","roll3","roll12","m","ano","logpop"]
tr = p[p.mes < "2025-01-01"]
te = p[p.mes >= "2025-01-01"].copy()
print(len(tr), "treino |", len(te), "teste")

## Modelos
**Atenção:** o Poisson do guia (sem padronizar) não converge: as colunas têm escalas muito diferentes (ano ≈ 2020, lags ≈ 0–10). Ele é mantido só para comparação. O modelo válido é o `poisson_pad`, com `StandardScaler`.

In [ ]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    po = PoissonRegressor(alpha=1e-3, max_iter=1000).fit(tr[feats], tr.y)
print("Poisson do guia - iterações:", po.n_iter_, "| aviso:", [str(x.message)[:60] for x in w])
te["poisson_guia"] = po.predict(te[feats])

po_pad = make_pipeline(StandardScaler(), PoissonRegressor(alpha=1e-3, max_iter=1000)).fit(tr[feats], tr.y)
te["poisson_pad"] = po_pad.predict(te[feats])

te["ingenuo"] = te["lag1"]
te["sazonal_ingenuo"] = te["lag12"]
te["media_12m"] = te["roll12"]
lr = LinearRegression().fit(tr[feats], tr.y)
te["linear"] = np.clip(lr.predict(te[feats]), 0, None)
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=42).fit(tr[feats], tr.y)
te["rf"] = rf.predict(te[feats])
te["zero"] = 0.0   # referência extra: prever sempre zero

## Tabela MAE/RMSE por grupo (mensal)

In [ ]:
grandes = p.groupby("cidade")["pop"].first().sort_values(ascending=False).head(10).index
print("10 maiores:", list(grandes))
te["grupo"] = np.where(te.cidade.isin(grandes), "grande", "pequeno")

mods = ["zero","ingenuo","sazonal_ingenuo","media_12m","linear","rf","poisson_pad","poisson_guia"]
linhas = []
for g, d in te.groupby("grupo"):
    for m in mods:
        linhas.append([g, m, mae(d.y, d[m]), mean_squared_error(d.y, d[m])**0.5, d[m].sum(), d.y.sum()])
tab = pd.DataFrame(linhas, columns=["grupo","modelo","MAE","RMSE","total_previsto","total_real"])
tab.to_csv("rq3_p2_metricas_mensal.csv", index=False)
tab.round(3)

## Tabela de resultados para o artigo (modelo, grupo, MAE, RMSE, viés)
Viés = média prevista ÷ média real − 1. Média real no teste: 0,764 (todos), 4,729 (grandes), 0,333 (pequenos).

In [ ]:
te["todos"] = "todos"
linhas = []
for col in ["grupo", "todos"]:
    for g, d in te.groupby(col):
        for m in ["media_12m", "linear", "rf", "poisson_pad"]:
            linhas.append([m, g, mae(d.y, d[m]), mean_squared_error(d.y, d[m])**0.5,
                           d[m].mean(), d.y.mean(), d[m].mean() / d.y.mean() - 1])
res = pd.DataFrame(linhas, columns=["modelo","grupo","MAE","RMSE","media_prevista","media_real","vies"])
res.round(4).to_csv("02_poisson_pollyana_resultados.csv", index=False)
res.round(3)

## Superdispersão
Razão de Pearson no treino: perto de 1 = sem superdispersão; acima de 1 = superdispersão.

In [ ]:
mu = po_pad.predict(tr[feats])
print("dispersão de Pearson:", round(((tr.y - mu)**2 / mu).sum() / (len(tr) - len(feats) - 1), 3))

## Teste trimestral (painel agregado por trimestre)
Os lags são refeitos no nível do trimestre, para não usar meses do próprio trimestre.

In [ ]:
p["tri"] = p.mes.dt.to_period("Q")
q = (p.groupby(["cidade","tri"]).agg(y=("y","sum"), pop=("pop","first"))
       .reset_index().sort_values(["cidade","tri"]))
g = q.groupby("cidade")["y"]
for k in (1, 2, 4):
    q[f"lag{k}"] = g.shift(k)
q["roll4"] = g.transform(lambda s: s.shift(1).rolling(4).mean())
q["t"] = q.tri.dt.quarter; q["ano"] = q.tri.dt.year; q["logpop"] = np.log(q["pop"])
q = q.dropna()
f = ["lag1","lag2","lag4","roll4","t","ano","logpop"]
qtr = q[q.tri < pd.Period("2025Q1")]; qts = q[q.tri >= pd.Period("2025Q1")].copy()

qts["zero"] = 0.0
qts["ingenuo"] = qts.lag1; qts["sazonal_ingenuo"] = qts.lag4; qts["media_4tri"] = qts.roll4
qts["linear"] = np.clip(LinearRegression().fit(qtr[f], qtr.y).predict(qts[f]), 0, None)
qts["rf"] = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1,
                                  random_state=42).fit(qtr[f], qtr.y).predict(qts[f])
qts["poisson_pad"] = make_pipeline(StandardScaler(), PoissonRegressor(alpha=1e-3, max_iter=1000)
                                   ).fit(qtr[f], qtr.y).predict(qts[f])
qts["grupo"] = np.where(qts.cidade.isin(grandes), "grande", "pequeno")

linhas = []
for gname, d in qts.groupby("grupo"):
    for m in ["zero","ingenuo","sazonal_ingenuo","media_4tri","linear","rf","poisson_pad"]:
        linhas.append([gname, m, mae(d.y, d[m]), mean_squared_error(d.y, d[m])**0.5])
tabq = pd.DataFrame(linhas, columns=["grupo","modelo","MAE","RMSE"])
tabq.to_csv("rq3_p2_metricas_trimestral.csv", index=False)
tabq.round(3)

## Gráfico real x previsto por grupo

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.9))
for a, g in zip(ax, ["grande", "pequeno"]):
    s = te[te.grupo == g].groupby("mes")[["y","media_12m","rf","poisson_pad","poisson_guia"]].sum()
    a.plot(s.index, s.y, color="black", lw=2.2, label="Real")
    a.plot(s.index, s.media_12m, color="#9AA5B1", label="Média 12 meses")
    a.plot(s.index, s.rf, color="#E39B2F", label="Random Forest")
    a.plot(s.index, s.poisson_pad, color="#2E5E8C", label="Poisson padronizado")
    a.plot(s.index, s.poisson_guia, color="#C0504D", ls="--", label="Poisson (código do guia)")
    a.set_ylim(0, None); a.set_title("10 maiores" if g == "grande" else "92 demais")
    a.tick_params(axis="x", rotation=30)
ax[1].legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.savefig("rq3_p2_previsto_real.png", dpi=150); plt.show()